In [1]:
import pandas as pd
import os
import re
import json
from collections import Counter

In [2]:
lookup_df = pd.read_csv('../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv')

def get_indonesian_column(filename):
    """Determines which column contains the Indonesian text based on the lookup table."""
    try:
        file_id = filename.split('_')[0].strip()
    except IndexError:
        return None

    for index, row in lookup_df.iterrows():
        raw_kamus = str(row['kamus'])
        match = re.search(r'\d+', raw_kamus)
        dict_id = match.group() if match else raw_kamus.strip()
            
        if dict_id == file_id:
            if int(row['is_from_indonesia']) == 1:
                return 'kalimat_asal'
            else:
                return 'kalimat_tujuan'
            
    print(f"Warning: No lookup found for {filename}")
    return None

In [3]:
def build_indonesian_dictionary(corpus_dir, dict_path, min_freq=2):
    word_counts = Counter()
    files = [f for f in os.listdir(corpus_dir) if f.endswith('.csv') and f != 'LookupIsFromIndonesia.csv']
    
    print(f"Found {len(files)} files. Extracting purely Indonesian words...")

    for file in files:
        target_col = get_indonesian_column(file)
        if not target_col:
            continue 
            
        filepath = os.path.join(corpus_dir, file)
        df = pd.read_csv(filepath)
        
        if target_col in df.columns:
            for text in df[target_col].dropna():
                words = re.findall(r'[a-z]+', str(text).lower())
                word_counts.update(words)

    freq_dict = {word: count for word, count in word_counts.items() if count >= min_freq}
    
    with open(dict_path, 'w') as f:
        json.dump(freq_dict, f)
            
    print(f"Dictionary built successfully with {len(freq_dict)} unique Indonesian words.")

corpus_path = '../Ekstraksi/11. Parallel Corpus'
dict_path = 'indonesian_dict.json'
build_indonesian_dictionary(corpus_path, dict_path)

Found 60 files. Extracting purely Indonesian words...
Dictionary built successfully with 61046 unique Indonesian words.
